# FABS Track 2 — RL Maze Agent · Colab Trainer

End-to-end notebook: clones repo (or uploads code), generates maps, trains PPO on T4, runs benchmark + ablation, downloads `viz/` and `agents/agent.zip`.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Expected wall-clock on T4 free: maps 30s · train 500k ≈ 75–110 min · benchmark 5 min · ablation (4× 100k) ≈ 50 min. Total ≈ 2–3 h.

## 1. Setup — pick ONE of the two cells below

In [ ]:
# Option A: clone from your GitHub (preferred — push the project there first)
!git clone https://github.com/Shalbulov/ai-hackathon-t2-maze-rl.git
%cd ai-hackathon-t2-maze-rl

In [ ]:
# Option B: zip & upload the project from local Mac, then unzip here
# (run on Mac:  cd ~/Downloads && zip -r ai-hackathon-t2-maze-rl.zip ai-hackathon-t2-maze-rl)
from google.colab import files  # noqa
uploaded = files.upload()  # select ai-hackathon-t2-maze-rl.zip
!unzip -q -o ai-hackathon-t2-maze-rl.zip
%cd ai-hackathon-t2-maze-rl

In [ ]:
!pip install -q -r requirements.txt
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Generate maps (4 train + 3 test, OOD-shifted)

In [ ]:
!python maze_gen.py --train 4 --test 3 --size 9 --seed 42
!ls maps/

## 3. Quick env smoke test (5 steps)

In [ ]:
from env import Maze3DEnv
import numpy as np
env = Maze3DEnv('maps/train1.npy', randomize=True)
obs, info = env.reset(seed=0)
assert obs.shape == (23,), obs.shape
for _ in range(5):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    print(f'step={env.steps} r={r:+.3f} pos={info["pos"]}')

## 4. Train (500k steps, ~1.5h on T4)

In [ ]:
!python train.py --steps 500000 --n-envs 4 --seed 42

## 5. Benchmark (100 ep × 7 maps + visualizations)

In [ ]:
!python benchmark.py --episodes 100
from IPython.display import Image, display
display(Image('viz/before_after.gif'))
display(Image('viz/heatmap_train1.png'))
print(open('viz/results_table.txt').read())

## 6. Ablation study (~50 min, optional but +8 rubric points)

In [ ]:
!python ablation.py --steps 100000 --episodes 20
print(open('viz/ablation.txt').read())

## 7. Download artifacts

In [ ]:
!zip -qr submission.zip agents/agent.zip viz/ maps/
from google.colab import files
files.download('submission.zip')

## 8. (Optional) Save to Google Drive so you don't lose it on disconnect

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fabs_t2
!cp -r agents viz maps /content/drive/MyDrive/fabs_t2/